# Gold — agregações e features

Parte do bundle Financial Services Medallion. Dados sintéticos.
DATASET_ID e CATALOG_NAME devem ser iguais em todas as tarefas.
Cada tarefa lê suas entradas das tabelas Delta persistidas, sem depender de variáveis de outra sessão.
Destinos existentes provocam falha para evitar sobrescrita; use novo DATASET_ID em uma nova carga.
Rótulos aleatórios: métricas de ML demonstrativas.


In [ ]:
import json, math, random, re
from datetime import datetime, timedelta
from decimal import Decimal
from uuid import uuid4
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DecimalType, IntegerType

if 'spark' not in globals() or 'oidlUtils' not in globals():
    raise RuntimeError('Execute no runtime AIDP com compute anexado.')

def parameter(name, default):
    value = oidlUtils.parameters.getParameter(name, str(default))
    return str(default) if value is None else str(value).strip()

def ident(s):
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', s):
        raise ValueError(f'Identificador inválido: {s!r}')
    return f'`{s}`'

catalog = parameter('CATALOG_NAME', 'default')
accounts = int(parameter('ACCOUNT_COUNT', 5000))
seed = int(parameter('SEED', 42))
num_trees = int(parameter('RF_NUM_TREES', 100))
max_depth = int(parameter('RF_MAX_DEPTH', 10))
liquid = parameter('ENABLE_LIQUID_CLUSTERING', 'false').lower() == 'true'
if not 100 <= accounts <= 5000 or not 1 <= num_trees <= 100 or not 1 <= max_depth <= 10:
    raise ValueError('Limites do demo: 100–5000 contas, 1–100 árvores, profundidade 1–10.')
run_id = parameter('DATASET_ID', 'bundle_dev')
if not re.fullmatch(r'[A-Za-z][A-Za-z0-9_]{0,47}', run_id):
    raise ValueError('DATASET_ID deve começar com letra e ter até 48 letras, números ou _.')
schemas = {layer: f'finance_{layer}_{run_id}' for layer in ('bronze', 'silver', 'gold')}
def table(layer, name):
    return '.'.join(ident(v) for v in (catalog, schemas[layer], name))
bronze_table = table('bronze', 'account_transactions_bronze')
silver_table = table('silver', 'account_transactions_silver')
analytics_table = table('gold', 'transaction_analytics_gold')
features_table = table('gold', 'fraud_detection_model_gold')
result = {'run_id': run_id, 'catalog': catalog, 'schemas': schemas, 'accounts': accounts, 'seed': seed, 'liquid_clustering_enabled': liquid}
def write_delta(frame, target, keys):
    if liquid:
        view = 'finance_stage_' + uuid4().hex[:8]
        frame.createOrReplaceTempView(view)
        spark.sql(f'CREATE TABLE {target} USING DELTA CLUSTER BY ({keys}) AS SELECT * FROM {view}')
        spark.catalog.dropTempView(view)
    else:
        frame.write.format('delta').mode('errorifexists').saveAsTable(target)

def ensure_schema(layer):
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {ident(catalog)}.{ident(schemas[layer])}')

def require_table(target):
    if not spark.catalog.tableExists(target):
        raise RuntimeError(f'Execute a camada anterior primeiro: {target}')


In [ ]:
ensure_schema('gold')
require_table(silver_table)
silver = spark.table(silver_table)
silver_count = silver.count()
# Gold: métricas mensais e distribuições de tipos de transação e estabelecimentos.
monthly = silver.withColumn('month_year', F.date_format('transaction_date', 'yyyy-MM'))
analytics = monthly.groupBy('account_id', 'month_year').agg(
    F.count('*').alias('total_transactions'), F.sum('amount').alias('total_amount'),
    F.avg('amount').alias('avg_amount'), F.avg('risk_score').alias('avg_risk_score'),
    F.sum(F.when(F.col('risk_score') > 60, 1).otherwise(0)).alias('high_risk_transactions'))
def distribution(column, output):
    return (monthly.groupBy('account_id', 'month_year', column).count()
        .groupBy('account_id', 'month_year')
        .agg(F.map_from_entries(F.collect_list(F.struct(F.col(column), F.col('count')))).alias(output)))
analytics = (analytics.join(distribution('transaction_type', 'transaction_types'), ['account_id', 'month_year'])
    .join(distribution('merchant_category', 'merchant_categories'), ['account_id', 'month_year'])
    .withColumn('created_timestamp', F.current_timestamp()))
write_delta(analytics, analytics_table, 'account_id, month_year')
gold_analytics = spark.table(analytics_table)
reconciled = gold_analytics.agg(F.sum('total_transactions').alias('transactions'), F.sum('total_amount').alias('amount')).first()
source_total = silver.agg(F.sum('amount').alias('amount')).first().amount
assert reconciled.transactions == silver_count
assert reconciled.amount == source_total
result['analytics_rows'] = gold_analytics.count()
result['amount_reconciled'] = True
ml_features = (silver.withColumn('is_fraud', F.when(F.col('risk_score') > 60, 1).otherwise(0))
    .select('account_id','transaction_date','transaction_type','amount','merchant_category','location','risk_score','month','day_of_week','hour','is_fraud')
    .withColumn('created_timestamp', F.current_timestamp()))
write_delta(ml_features, features_table, 'account_id, transaction_date')
ml_features = spark.table(features_table)
assert ml_features.count() == silver_count
result['ml_feature_rows'] = silver_count
print('FINANCE_STAGE=gold_complete ' + json.dumps(result, default=str))

In [ ]:

result.update(stage='gold', status='SUCCESS', synthetic_labels_only=True)
result['tables'] = [t.replace('`', '') for t in (analytics_table,features_table,)]
print('AIDP_FINANCE_BUNDLE_STAGE_RESULT=' + json.dumps(result, sort_keys=True))
